# Cell 1: Library Setup & Recursive Dataset Crawler

In [1]:
import os
import cv2
import glob
import shutil
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Define exact Kaggle dataset path
DATASET_PATH = "/kaggle/input/datasets/ari7889/bangladesh-traffic/Bangladeshi Traffic Flow Dataset"
ANNOTATED_DIR = os.path.join(DATASET_PATH, "Annotated Images")

image_paths = []
xml_paths = []

print("Crawling through directories to find images and XML annotations...")

# Recursively dig through all nested sub-folders
for root, dirs, files in os.walk(ANNOTATED_DIR):
    for file in files:
        if file.endswith(('.jpg', '.jpeg', '.png')):
            base_name = os.path.splitext(file)[0]
            expected_xml = base_name + '.xml'
            
            if expected_xml in files:
                image_paths.append(os.path.join(root, file))
                xml_paths.append(os.path.join(root, expected_xml))

print(f"Success! Total valid image-annotation pairs found: {len(image_paths)}")

Crawling through directories to find images and XML annotations...
Success! Total valid image-annotation pairs found: 23678


# Cell 2: Data Splitting & Restructuring

In [2]:
# Kaggle's writable directory where we will store the finalized dataset
BASE_OUT_DIR = "/kaggle/working/tfp_dataset" 

print("Splitting data into Train (70%), Validation (20%), and Test (10%)...")

train_val_img, test_img, train_val_xml, test_xml = train_test_split(
    image_paths, xml_paths, test_size=0.10, random_state=42
)

train_img, val_img, train_xml, val_xml = train_test_split(
    train_val_img, train_val_xml, test_size=0.2222, random_state=42
)

print(f"Split distribution -> Train: {len(train_img)}, Validation: {len(val_img)}, Test: {len(test_img)}")

splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(BASE_OUT_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(BASE_OUT_DIR, split, 'labels'), exist_ok=True)

def copy_files(img_list, xml_list, split_name):
    print(f"Copying {split_name} files...")
    for img_src, xml_src in tqdm(zip(img_list, xml_list), total=len(img_list)):
        img_name = os.path.basename(img_src)
        xml_name = os.path.basename(xml_src)
        
        img_dst = os.path.join(BASE_OUT_DIR, split_name, 'images', img_name)
        xml_dst = os.path.join(BASE_OUT_DIR, split_name, 'labels', xml_name)
        
        shutil.copy(img_src, img_dst)
        shutil.copy(xml_src, xml_dst)

copy_files(train_img, train_xml, 'train')
copy_files(val_img, val_xml, 'val')
copy_files(test_img, test_xml, 'test')
print("\nData splitting and transfer complete!")

Splitting data into Train (70%), Validation (20%), and Test (10%)...
Split distribution -> Train: 16574, Validation: 4736, Test: 2368
Copying train files...


100%|██████████| 16574/16574 [02:38<00:00, 104.44it/s]


Copying val files...


100%|██████████| 4736/4736 [00:45<00:00, 104.08it/s]


Copying test files...


100%|██████████| 2368/2368 [00:22<00:00, 104.40it/s]


Data splitting and transfer complete!


# Cell 3: Case-Insensitive YOLO Conversion Logic (THE FIX)

In [3]:
import xml.etree.ElementTree as ET

# Your officially proposed classes (Updated to include Mini-Truck instead of Van/Leguna)
CLASSES = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle", "Rickshaw", "Mini-Truck", "CNG", "Pedestrian"]

# THE FIX: A translation dictionary that maps the dataset's weird labels to your official labels
LABEL_MAPPING = {
    "car": "Car",
    "bus": "Bus",
    "truck": "Truck",
    "bike": "Motorcycle",     # Translates 'Bike' to 'Motorcycle'
    "cycle": "Bicycle",       # Translates 'Cycle' to 'Bicycle'
    "rickshaw": "Rickshaw",
    "mini-truck": "Mini-Truck", 
    "cng": "CNG",
    "people": "Pedestrian"    # Translates 'People' to 'Pedestrian'
}

def convert_voc_to_yolo(xml_path, txt_path):
    """Reads a VOC XML, translates the label, normalizes the coordinates, and writes a YOLO TXT file."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    size = root.find('size')
    if size is not None:
        img_w = float(size.find('width').text)
        img_h = float(size.find('height').text)
    else:
        return False 

    with open(txt_path, 'w') as out_file:
        for obj in root.findall('object'):
            raw_name = obj.find('name').text
            
            # Clean up the raw XML name and make it lowercase
            safe_name = str(raw_name).strip().lower()
            
            # If the raw name isn't in our mapping dictionary, skip it
            if safe_name not in LABEL_MAPPING:
                continue
            
            # Get your official class name, then find its ID number (0-8)
            official_class = LABEL_MAPPING[safe_name]
            class_id = CLASSES.index(official_class)
            
            xmlbox = obj.find('bndbox')
            xmin = float(xmlbox.find('xmin').text)
            ymin = float(xmlbox.find('ymin').text)
            xmax = float(xmlbox.find('xmax').text)
            ymax = float(xmlbox.find('ymax').text)
            
            x_center = ((xmin + xmax) / 2) / img_w
            y_center = ((ymin + ymax) / 2) / img_h
            box_w = (xmax - xmin) / img_w
            box_h = (ymax - ymin) / img_h
            
            out_file.write(f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}\n")
            
    return True

print("Translation dictionary and YOLO conversion logic updated successfully!")

Translation dictionary and YOLO conversion logic updated successfully!


# Cell 4: Execute the Conversion & Clean Up

In [4]:
dataset_base = "/kaggle/working/tfp_dataset"
splits = ['train', 'val', 'test']

for split in splits:
    label_dir = os.path.join(dataset_base, split, 'labels')
    print(f"\nConverting {split.upper()} annotations to YOLO format...")
    
    xml_files = [f for f in os.listdir(label_dir) if f.endswith('.xml')]
    
    for xml_file in tqdm(xml_files):
        xml_path = os.path.join(label_dir, xml_file)
        txt_path = os.path.join(label_dir, xml_file.replace('.xml', '.txt'))
        
        success = convert_voc_to_yolo(xml_path, txt_path)
        
        if success:
            os.remove(xml_path)
            
print("\nSuccess! All Pascal VOC annotations have been converted to YOLO format.")


Converting TRAIN annotations to YOLO format...


100%|██████████| 16574/16574 [00:04<00:00, 3862.42it/s]



Converting VAL annotations to YOLO format...


100%|██████████| 4736/4736 [00:01<00:00, 3908.69it/s]



Converting TEST annotations to YOLO format...


100%|██████████| 2368/2368 [00:00<00:00, 3955.28it/s]


Success! All Pascal VOC annotations have been converted to YOLO format.
